In [1]:
from transformers import pipeline
import pandas as pd
import os


os.chdir("/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/Data")
data = pd.read_csv("MN_Health_short.csv")

text = data["Text"].str.slice(0,50)
lst = text.to_list()

hypothesis_template = "This text is about {}"
classes_verbalized = ["covid", "other"]

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

output = zeroshot_classifier(lst, classes_verbalized, hypothesis_template=hypothesis_template, multi_label=False)

clean_output = []

for dct in output:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

/opt/homebrew/Caskroom/miniconda/base/envs/torch-gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Device set to use mps


In [2]:
df

,sequence,covid,other
0,The Minnesota Department of Health today annou...,0.999999,1.452811e-06
1,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05
2,This case marks the first documented instance ...,0.646615,3.533848e-01
3,A rapidly growing outbreak of variant COVID-19...,0.999999,1.194613e-06
4,As health officials continue to report more ca...,0.999999,9.713034e-07
5,Reflecting ongoing concerns about the potentia...,0.770522,2.294779e-01
6,May 12 will be the last day of testing at the ...,0.999999,7.824493e-07
7,Participating health plans starting this work ...,0.191590,8.084098e-01
8,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05


In [5]:

output2 = zeroshot_classifier(lst, classes_verbalized, 
                              hypothesis_template=hypothesis_template, 
                              multi_label=True)

clean_output = []

for dct in output2:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df2 = pd.DataFrame(clean_output)


In [32]:
from scipy.special import softmax

df2[["covid", "other"]].apply(lambda row: softmax(row), axis = 1)
# softmax(df2.to_numpy(, axis = 1)
# df2.apply(lambda row: softmax(row), axis = 1)


0     [0.7310567712812766, 0.2689432287187235]
1    [0.7310449227902115, 0.26895507720978845]
2    [0.5000112290304041, 0.49998877096959593]
3     [0.7310541765610605, 0.2689458234389394]
4     [0.7310554719397776, 0.2689445280602224]
5     [0.5001616503557739, 0.4998383496442262]
6     [0.7310567685503638, 0.2689432314496361]
7           [0.4999935409069, 0.5000064590931]
8    [0.7310449227902115, 0.26895507720978845]
dtype: object

In [33]:
df

,sequence,covid,other
0,The Minnesota Department of Health today annou...,0.999999,1.452811e-06
1,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05
2,This case marks the first documented instance ...,0.646615,3.533848e-01
3,A rapidly growing outbreak of variant COVID-19...,0.999999,1.194613e-06
4,As health officials continue to report more ca...,0.999999,9.713034e-07
5,Reflecting ongoing concerns about the potentia...,0.770522,2.294779e-01
6,May 12 will be the last day of testing at the ...,0.999999,7.824493e-07
7,Participating health plans starting this work ...,0.191590,8.084098e-01
8,The Minnesota Department of Health (MDH) today...,0.999983,1.653233e-05
